<a href="https://colab.research.google.com/github/parbin-12/Minor_project_Retinal_image_segmentation/blob/main/minor_project_phase_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision.transforms import transforms
from PIL import Image


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

print(os.listdir('/content/drive/MyDrive/DRIVE'))

In [ ]:
#training path
print(os.listdir('/content/drive/MyDrive/DRIVE/training'))
image_dir = os.path.join(base_path, 'images')
mask_dir = os.path.join(base_path, '1st_manual')

In [ ]:
base_path = '/content/drive/MyDrive/DRIVE/training'
for folder in ['images' , '1st_manual' , 'mask']:
  path = os.path.join(base_path,folder)
  print(folder ,":",len(os.listdir(path)))
  print(os.listdir(path)[:5])
  print()

In [ ]:
#opening the image
image = Image.open('/content/drive/MyDrive/DRIVE/training/images/33_training.tif')

#information
print("origional image",image.size)
print("origional image",image.mode)
# resize
resized = image.resize((256, 256))
print("New size:", resized.size)
# display
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(resized)
plt.title("256 x 256")
plt.axis("off")

plt.show()

In [ ]:
#image transformation is defined here

image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

#doing the transformation for the mask image
mask_transform = transforms.Compose([
    transforms.Resize((256,256), interpolation = transforms.InterpolationMode.NEAREST),
    transforms.ToTensor()
])

In [ ]:
#defining our custom dataset now
class CustomDataset(Dataset):
  def __init__(self,image_dir,mask_dir,image_transform = None ,mask_transform = None):
    self.image_dir = image_dir
    self.mask_dir = mask_dir
    self.image_transform = image_transform
    self.mask_transform = mask_transform
    self.images = os.listdir(image_dir)

  def __len__(self):
    return len(self.images)

  def __getitem__(self, index):

        image_name = self.images[index]
        patient_id = image_name.split('_')[0]
        mask_name = patient_id + '_manual1.gif'
        image_path = os.path.join(
            self.image_dir,
            image_name
        )
        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.image_transform:
            image = self.image_transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)
        return image, mask


In [ ]:
dataset = CustomDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    image_transform=image_transform,
    mask_transform=mask_transform
)

In [ ]:
image,mask = dataset[0]
print("image shape",image.shape)
print("mask shape",mask.shape)

In [ ]:
#here is the dataloader now
train_data = DataLoader(
    dataset,
    batch_size = 4,
    shuffle = True
)

In [ ]:
images, masks = next(iter(train_data))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(4):

    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title(f"image {i+1}")
    plt.axis("off")

    plt.subplot(2, 4, i + 5)
    plt.imshow(masks[i].squeeze(), cmap="gray")
    plt.title(f"mask {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()